# 04 - Blue Zone Country Deep Dives

Detailed analysis of each Blue Zone country compared to its regional peers.

**Important limitation:** Blue Zones are specific regions within countries (Okinawa, Sardinia,
Ikaria, Nicoya, Loma Linda), not entire countries. This analysis tracks country-level data
because sub-national historical time series are not available from public APIs.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'historical', 'merged_historical_panel.csv'))
profiles = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'blue_zone_country_profiles.csv'))

REGIONS = {
    'East Asia': ['JPN', 'KOR', 'CHN', 'SGP', 'THA', 'VNM', 'MYS', 'IDN', 'PHL', 'MMR'],
    'Western Europe': ['ITA', 'FRA', 'DEU', 'ESP', 'GBR', 'NLD', 'BEL', 'AUT', 'CHE', 'PRT'],
    'Southern Europe': ['GRC', 'ITA', 'ESP', 'PRT', 'ALB', 'HRV', 'SVN'],
    'Central America': ['CRI', 'MEX', 'COL', 'VEN', 'ECU', 'PER'],
    'North America': ['USA', 'CAN', 'MEX'],
}

BZ_INFO = {
    'JPN': {'name': 'Japan', 'region': 'Okinawa', 'peer_region': 'East Asia'},
    'ITA': {'name': 'Italy', 'region': 'Sardinia', 'peer_region': 'Western Europe'},
    'GRC': {'name': 'Greece', 'region': 'Ikaria', 'peer_region': 'Southern Europe'},
    'CRI': {'name': 'Costa Rica', 'region': 'Nicoya Peninsula', 'peer_region': 'Central America'},
    'USA': {'name': 'United States', 'region': 'Loma Linda, CA', 'peer_region': 'North America'},
}

print('Data loaded successfully')

Data loaded successfully


## Deep Dive: Each Blue Zone Country

In [2]:
fig, axes = plt.subplots(3, 2, figsize=(18, 18))
axes = axes.flatten()

for i, (iso, info) in enumerate(BZ_INFO.items()):
    ax = axes[i]
    
    # Country data
    country = df[(df['iso_code'] == iso) & (df['life_expectancy'].notna())].sort_values('year')
    
    # Peer region data
    peer_isos = [c for c in REGIONS.get(info['peer_region'], []) if c != iso]
    peer_data = df[(df['iso_code'].isin(peer_isos)) & (df['life_expectancy'].notna())]
    peer_avg = peer_data.groupby('year')['life_expectancy'].agg(['mean', 'min', 'max']).reset_index()
    
    # Global average
    global_avg = df[df['life_expectancy'].notna()].groupby('year')['life_expectancy'].mean().reset_index()
    
    # Plot
    ax.plot(country['year'], country['life_expectancy'], 'b-', linewidth=2.5,
            label=f'{info["name"]}')
    if len(peer_avg) > 0:
        ax.plot(peer_avg['year'], peer_avg['mean'], color='orange', linewidth=1.5,
                linestyle='--', label=f'{info["peer_region"]} avg')
        ax.fill_between(peer_avg['year'], peer_avg['min'], peer_avg['max'],
                       alpha=0.1, color='orange', label=f'{info["peer_region"]} range')
    ax.plot(global_avg['year'], global_avg['life_expectancy'], 'gray', linewidth=1,
            linestyle=':', alpha=0.7, label='Global avg')
    
    ax.set_title(f'{info["name"]} (Blue Zone: {info["region"]})', fontsize=13)
    ax.set_xlabel('Year')
    ax.set_ylabel('Life Expectancy (years)')
    ax.legend(fontsize=8, loc='lower right')
    ax.set_xlim(1960, 2023)
    
    # Stats annotation
    if len(country) > 0:
        le = country['life_expectancy']
        improvement = le.iloc[-1] - le.iloc[0]
        ax.text(0.02, 0.98, f'Total gain: +{improvement:.1f} yr\nLatest: {le.iloc[-1]:.1f} yr',
               transform=ax.transAxes, fontsize=9, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# Remove empty subplot
axes[5].set_visible(False)

fig.suptitle('Blue Zone Countries vs Regional Peers (1960-2023)', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb04_country_deep_dives.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb04_country_deep_dives.png')

Saved: nb04_country_deep_dives.png


/tmp/ipykernel_1567876/1068268778.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Multi-Indicator Comparison

In [3]:
# Compare BZ countries on multiple indicators (most recent data)
indicators = ['life_expectancy', 'gdp_per_capita', 'physicians_per_1000',
              'health_expenditure_pc', 'urban_population_pct']

bz_recent = df[(df['is_blue_zone'] == 1) & (df['year'] >= 2018)]
bz_summary = bz_recent.groupby('iso_code')[indicators].mean().round(1)

# Add country names
bz_summary.index = [BZ_INFO[iso]['name'] for iso in bz_summary.index]
bz_summary

,life_expectancy,gdp_per_capita,physicians_per_1000,health_expenditure_pc,urban_population_pct
Costa Rica,79.7,13545.8,2.5,989.5,77.7
Greece,81.2,20330.1,6.3,1737.3,78.4
Italy,83.0,35432.0,4.0,3142.0,69.5
Japan,84.3,38032.0,2.6,4327.1,91.8
United States,77.8,69776.1,3.4,11753.1,80.1


In [4]:
# GDP vs Life Expectancy trajectory for each BZ country
fig, ax = plt.subplots(figsize=(14, 8))

colors = {'JPN': '#E74C3C', 'ITA': '#2ECC71', 'GRC': '#9B59B6', 'CRI': '#F39C12', 'USA': '#3498DB'}

for iso, info in BZ_INFO.items():
    c = df[(df['iso_code'] == iso) & (df['gdp_per_capita'].notna()) &
           (df['life_expectancy'].notna())].sort_values('year')
    if len(c) > 0:
        ax.plot(c['gdp_per_capita'], c['life_expectancy'], '-o', color=colors[iso],
                markersize=3, alpha=0.7, label=info['name'])
        # Label start and end
        ax.annotate(f"{info['name']} {int(c['year'].iloc[0])}",
                   (c['gdp_per_capita'].iloc[0], c['life_expectancy'].iloc[0]),
                   fontsize=7, alpha=0.7)
        ax.annotate(f"{int(c['year'].iloc[-1])}",
                   (c['gdp_per_capita'].iloc[-1], c['life_expectancy'].iloc[-1]),
                   fontsize=8, fontweight='bold')

ax.set_xlabel('GDP per Capita (USD)', fontsize=12)
ax.set_ylabel('Life Expectancy (years)', fontsize=12)
ax.set_title('GDP vs Life Expectancy Trajectory for Blue Zone Countries', fontsize=14)
ax.legend(fontsize=10)
ax.set_xscale('log')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb04_gdp_vs_le_trajectory.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb04_gdp_vs_le_trajectory.png')

Saved: nb04_gdp_vs_le_trajectory.png


/tmp/ipykernel_1567876/2375633332.py:29: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Blue Zone Country Rankings Over Time

In [5]:
# Calculate global rank for each BZ country per year
le_data = df[df['life_expectancy'].notna()].copy()
le_data['rank'] = le_data.groupby('year')['life_expectancy'].rank(ascending=False)
le_data['n_total'] = le_data.groupby('year')['life_expectancy'].transform('count')
le_data['percentile'] = (1 - le_data['rank'] / le_data['n_total']) * 100

fig, ax = plt.subplots(figsize=(14, 7))

for iso, info in BZ_INFO.items():
    c = le_data[le_data['iso_code'] == iso].sort_values('year')
    if len(c) > 0:
        ax.plot(c['year'], c['percentile'], '-', color=colors[iso],
                linewidth=2, label=f"{info['name']}")

ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Life Expectancy Percentile (higher = better)', fontsize=12)
ax.set_title('Blue Zone Country Rankings Over Time (Percentile among all countries)', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 105)
ax.axhline(y=50, color='gray', linestyle=':', alpha=0.5, label='Median')
ax.set_xlim(1960, 2023)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb04_bz_rankings.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb04_bz_rankings.png')

Saved: nb04_bz_rankings.png


/tmp/ipykernel_1567876/1694408317.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Country-Level Summary

In [6]:
for iso, info in BZ_INFO.items():
    c = df[(df['iso_code'] == iso) & (df['life_expectancy'].notna())].sort_values('year')
    le = c['life_expectancy']
    print(f"\n{info['name']} (Blue Zone: {info['region']})")
    print(f"  Period: {c['year'].min()}-{c['year'].max()}")
    print(f"  LE start: {le.iloc[0]:.1f} years")
    print(f"  LE latest: {le.iloc[-1]:.1f} years")
    print(f"  Total gain: +{le.iloc[-1] - le.iloc[0]:.1f} years")
    print(f"  Avg gain per decade: +{(le.iloc[-1] - le.iloc[0]) / ((c['year'].max() - c['year'].min()) / 10):.1f} years")
    
    # Ranking
    latest_rank = le_data[(le_data['iso_code'] == iso) & (le_data['year'] >= 2020)]
    if len(latest_rank) > 0:
        r = latest_rank.iloc[-1]
        print(f"  Current rank: #{int(r['rank'])} out of {int(r['n_total'])} (top {r['percentile']:.0f}%)")


Japan (Blue Zone: Okinawa)
  Period: 1960-2023
  LE start: 67.7 years
  LE latest: 84.0 years
  Total gain: +16.3 years
  Avg gain per decade: +2.6 years
  Current rank: #2 out of 93 (top 98%)

Italy (Blue Zone: Sardinia)
  Period: 1960-2023
  LE start: 69.1 years
  LE latest: 83.7 years
  Total gain: +14.6 years
  Avg gain per decade: +2.3 years
  Current rank: #4 out of 93 (top 96%)

Greece (Blue Zone: Ikaria)
  Period: 1960-2023
  LE start: 70.4 years
  LE latest: 81.5 years
  Total gain: +11.1 years
  Avg gain per decade: +1.8 years
  Current rank: #24 out of 93 (top 74%)

Costa Rica (Blue Zone: Nicoya Peninsula)
  Period: 1960-2023
  LE start: 63.5 years
  LE latest: 80.8 years
  Total gain: +17.3 years
  Avg gain per decade: +2.8 years
  Current rank: #27 out of 93 (top 71%)

United States (Blue Zone: Loma Linda, CA)
  Period: 1960-2023
  LE start: 69.8 years
  LE latest: 78.4 years
  Total gain: +8.6 years
  Avg gain per decade: +1.4 years
  Current rank: #35 out of 93 (top 62%